> **Experiment A: Isolated, one model per event**  
> Spec, decision rule and results: [docs/experiments/A_isolated_per_event.md](../docs/experiments/A_isolated_per_event.md)  
> Depends on: E00 · Status: done-unverified · Config: [configs/arm_isolated.yaml](../configs/arm_isolated.yaml)  
> Registered in `experiments/registry.yaml`; see the [experiment graph](../docs/experiments/EXPERIMENT_GRAPH.md).

# CARE to Compare — Wind Farm C — **Arm A: event-isolated**

One autoencoder per event dataset. Each model's training rows, scaler, early-stopping split and threshold come from that event's own training partition and nothing else.

This is the paper's structure (*"individual autoencoder models per turbine"*) at the strictest granularity, and the only configuration in which the dataset's cross-file row duplication cannot reach the evaluation.

**Setup:** Wind Farm C · Avg features only · paper Table 3 architecture `[133, 83, 20, 83, 133]` · threshold = 99th percentile of reconstruction error on held-out normal rows · criticality threshold 72.

Run this first, then Arm B, then the comparison cell at the end of Arm B.

## Shared code
Identical to Arm B — any difference in results must come from the training strategy, not from preprocessing or scoring.

In [ ]:
# =============================================================================
# CONFIG  --  identical in both notebooks. Do not change one without the other.
# =============================================================================
import os, gc, re, json, math, time, hashlib, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# ---- scope -----------------------------------------------------------------
FARM               = "Wind Farm C"     # FAQ: status labels are only evaluation-
                                       # usable for Farms B and C. Farm A differs.

# ---- features (FAQ: "Avg features are often the safest starting point";
#      Min/Max/Std contain implausible values) ---------------------------------
USE_AVG_ONLY       = True
DROP_REACTIVE      = False             # set True to drop reactive_power_* sensors
ANGLE_TRANSFORM    = True              # FAQ lists angle transformation as a
                                       # typical preprocessing step
MAX_NAN_FRACTION   = 0.30              # drop a feature if >30% NaN in training

# ---- normal-behaviour filter (FAQ: filtering by status_type_id is intended
#      use; power-curve filtering is "also reasonable") -----------------------
NORMAL_STATUS_IDS      = {0, 2}
WIND_SPEED_CUTIN       = 3.0
WIND_SPEED_RATED       = 12.0
POWER_NEAR_ZERO_THRESH = 0.02

# ---- model (paper Table 3, Wind Farm C) ------------------------------------
HIDDEN_LAYERS      = [133, 83, 20, 83, 133]
LEARNING_RATE      = 0.0056            # paper Table 3
USE_LR_PLATEAU     = True
# At the paper's LR the TRAINING loss increases across epochs while validation
# plateaus -- the optimiser is overshooting, and early stopping then restores an
# under-trained model. ReduceLROnPlateau keeps the paper's starting LR but lets
# it decay when validation stalls. Set False to reproduce the paper exactly.
LR_PLATEAU_FACTOR  = 0.5
LR_PLATEAU_PATIENCE = 4
NOISE_STD          = 0.0               # no denoising noise for Farm C
BATCH_SIZE         = 64
MAX_EPOCHS         = 200
EARLY_STOP_PATIENCE = 10               # deviation from the 3 used earlier:
                                       # 3 stops on noise before convergence
VAL_FRACTION       = 0.15              # held out for early stopping AND for
                                       # threshold calibration (never trained on)
VAL_SPLIT_MODE     = "blocked"         # "blocked" | "chronological" | "random"
VAL_BLOCK_HOURS    = 120               # 5-day blocks for the blocked split
# RANDOM interleaves validation rows between training rows minutes apart, so
#   validation error measures within-period noise only. The threshold comes out
#   far too low for a prediction window months later, and the model over-flags.
# CHRONOLOGICAL takes the last VAL_FRACTION of the training period. Safe on
#   pooled data, but on a SINGLE event (~37k rows, ~8 months) that tail is one
#   contiguous season the model never saw, so validation error explodes and the
#   threshold with it -- observed spread of 42x across events.
# BLOCKED splits the period into VAL_BLOCK_HOURS blocks and takes every k-th one.
#   Validation rows are never adjacent to training rows (no autocorrelation
#   leakage) AND they span the full seasonal range (no regime shift). This is the
#   default for that reason.
SEED               = 42

# ---- thresholding ----------------------------------------------------------
# FAQ: "for early fault detection, you want to predict the event_label" -- NOT
# status_type_id. A normal-behaviour model has no anomaly labels at fit time, so
# the threshold is a high quantile of reconstruction error on held-out NORMAL
# data. Sweep SCORE_QUANTILE to trade Coverage against Accuracy.
SCORE_QUANTILE        = 0.99
CRITICALITY_THRESHOLD = 72             # FAQ: the documented default
BETA                  = 0.5
CARE_WEIGHTS          = (1, 1, 1, 2)   # (Coverage, Earliness, Reliability, Accuracy)

DTYPE = np.float32   # sensor dtype, used by every reader

np.random.seed(SEED)
print("Config loaded. Farm:", FARM)

In [ ]:
# =============================================================================
# DATA DISCOVERY
# =============================================================================
DATASET_ROOT = "/kaggle/input/datasets/azizkasimov/wind-turbine-scada-data-for-early-fault-detection"

def find_farm_dir(root, farm_name):
    for dirpath, dirnames, filenames in os.walk(root):
        if os.path.basename(dirpath) == farm_name and "datasets" in dirnames:
            return dirpath
    raise FileNotFoundError(f"Could not locate '{farm_name}' under {root}")

FARM_DIR     = find_farm_dir(DATASET_ROOT, FARM)
DATASETS_DIR = os.path.join(FARM_DIR, "datasets")

def first_existing(*cands):
    for c in cands:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(cands)

event_info = pd.read_csv(first_existing(
    os.path.join(FARM_DIR, "comma_event_info.csv"),
    os.path.join(FARM_DIR, "event_info.csv")))
feature_description = pd.read_csv(first_existing(
    os.path.join(FARM_DIR, "comma_feature_description.csv"),
    os.path.join(FARM_DIR, "feature_description.csv")))

event_info["event_start"] = pd.to_datetime(event_info["event_start"], errors="coerce")
event_info["event_end"]   = pd.to_datetime(event_info["event_end"],   errors="coerce")

# The dataset ships each file twice (comma- and semicolon-delimited). Pick one
# variant so no event is loaded twice.
all_csvs = [f for f in os.listdir(DATASETS_DIR) if f.endswith(".csv")]
has_comma = any(f.startswith("comma_") for f in all_csvs)
has_plain = any(not f.startswith("comma_") for f in all_csvs)
if has_comma and has_plain:
    csv_files, DATA_SEP = [f for f in all_csvs if f.startswith("comma_")], ","
else:
    csv_files, DATA_SEP = all_csvs, ("," if has_comma else ";")

def extract_dataset_id(fname):
    m = re.search(r"\d+", fname)
    if not m:
        raise ValueError(f"No numeric id in {fname}")
    return int(m.group())

csv_files = sorted(csv_files, key=extract_dataset_id)
ids = [extract_dataset_id(f) for f in csv_files]
assert len(ids) == len(set(ids)), "Duplicate dataset ids after variant filtering"

n_anom   = (event_info["event_label"] == "anomaly").sum()
n_normal = (event_info["event_label"] == "normal").sum()
print(f"{len(csv_files)} event datasets | event_info: {n_anom} anomaly / {n_normal} normal")

# asset_id -> which physical turbine each event belongs to. Needed for the
# leakage diagnostics and for the pooled notebook's asset-disjoint split.
# Farm C's event_info has no asset_id column, but every DATA file carries one.
# Read a single cell from each file -- cheap, and without it we cannot tell
# which events share a turbine.
def build_event_to_asset():
    if "asset_id" in event_info.columns:
        return dict(zip(event_info["event_id"].astype(int),
                        event_info["asset_id"].astype(int)))
    m = {}
    for f in csv_files:
        try:
            v = pd.read_csv(os.path.join(DATASETS_DIR, f), sep=DATA_SEP,
                            usecols=["asset_id"], nrows=1)["asset_id"].iloc[0]
            m[extract_dataset_id(f)] = int(v)
        except Exception as e:
            print(f"  could not read asset_id from {f}: {e}")
    return m

EVENT_TO_ASSET = build_event_to_asset()
if EVENT_TO_ASSET:
    n_assets = len(set(EVENT_TO_ASSET.values()))
    print(f"{n_assets} distinct turbines across {len(csv_files)} events "
          f"({len(csv_files) - n_assets} events share a turbine with another event)")
else:
    print("WARNING: no asset_id anywhere -- effective-N and the pooled split "
          "will fall back to event-level handling")

In [ ]:
# =============================================================================
# FEATURE SELECTION  --  Avg-only, optional angle transform
# =============================================================================
DESCRIPTIVE_COLS = ["time_stamp", "asset_id", "id", "train_test", "status_type_id"]

_sample = pd.read_csv(os.path.join(DATASETS_DIR, csv_files[0]), sep=DATA_SEP, nrows=200)
ALL_SENSOR_COLS = [c for c in _sample.columns if c not in DESCRIPTIVE_COLS]

if USE_AVG_ONLY:
    SENSOR_COLS = [c for c in ALL_SENSOR_COLS if c.endswith("_avg")]
else:
    SENSOR_COLS = list(ALL_SENSOR_COLS)

if DROP_REACTIVE:
    SENSOR_COLS = [c for c in SENSOR_COLS if not c.startswith("reactive_power_")]

print(f"{len(ALL_SENSOR_COLS)} sensor columns -> {len(SENSOR_COLS)} selected")

# ---- angle detection -------------------------------------------------------
# Prefer metadata; fall back to name prefixes. Always print what was found so
# the choice can be checked rather than trusted.
def detect_angle_sensors(feature_description, sensor_cols):
    angle_bases = set()
    text_cols = [c for c in feature_description.columns
                 if feature_description[c].dtype == object and c != "sensor_name"]
    pat = re.compile(r"\b(degree|degrees|deg|angle|direction|azimuth)\b", re.I)
    for _, row in feature_description.iterrows():
        blob = " ".join(str(row[c]) for c in text_cols)
        if pat.search(blob):
            angle_bases.add(str(row["sensor_name"]))
    if not angle_bases:  # metadata gave nothing -> name-based fallback
        name_pat = re.compile(r"^(wind_direction|nacelle_direction|nacelle_position|"
                              r"pitch_angle|blade_angle|yaw_angle|wind_dir)", re.I)
        angle_bases = {n for n in feature_description["sensor_name"].astype(str)
                       if name_pat.match(n)}
    angle_cols = [c for c in sensor_cols
                  if any(c == f"{b}_avg" or c.startswith(f"{b}_") for b in angle_bases)]
    return sorted(set(angle_cols))

ANGLE_COLS = detect_angle_sensors(feature_description, SENSOR_COLS) if ANGLE_TRANSFORM else []
LINEAR_COLS = [c for c in SENSOR_COLS if c not in set(ANGLE_COLS)]
print(f"Angle columns detected ({len(ANGLE_COLS)}): {ANGLE_COLS if len(ANGLE_COLS) <= 12 else ANGLE_COLS[:12] + ['...']}")

# Sanity-check the detection: a true angle column spans roughly 0-360 (or
# -180..180). Anything outside that is a false positive and must NOT be sin/cos'd.
if ANGLE_COLS:
    _chk = pd.read_csv(os.path.join(DATASETS_DIR, csv_files[0]), sep=DATA_SEP,
                       usecols=ANGLE_COLS)
    _lo, _hi = _chk.min(), _chk.max()
    _suspect = [c for c in ANGLE_COLS
                if not ((-190 <= _lo[c] <= 5) and (170 <= _hi[c] <= 370))]
    print("\nAngle range check (first file):")
    for c in ANGLE_COLS:
        flag = "  <-- NOT angle-like" if c in _suspect else ""
        print(f"  {c:<22} [{_lo[c]:8.2f}, {_hi[c]:8.2f}]{flag}")
    if _suspect:
        print(f"\nDropping {len(_suspect)} false positives from the angle set.")
        ANGLE_COLS = [c for c in ANGLE_COLS if c not in _suspect]
        LINEAR_COLS = [c for c in SENSOR_COLS if c not in set(ANGLE_COLS)]
    del _chk

FEATURE_NAMES = LINEAR_COLS + [f"{c}__sin" for c in ANGLE_COLS] + [f"{c}__cos" for c in ANGLE_COLS]
N_FEATURES = len(FEATURE_NAMES)
print(f"Model input dimension: {N_FEATURES}")

PRESENT_DESCRIPTIVE = [c for c in DESCRIPTIVE_COLS if c in _sample.columns]
READ_COLS = PRESENT_DESCRIPTIVE + SENSOR_COLS
print(f"Columns actually parsed per file: {len(READ_COLS)} of {len(_sample.columns)}")

POWER_COL = next((f"{b}_avg" for b in feature_description["sensor_name"]
                  if str(b).startswith("power_") and f"{b}_avg" in SENSOR_COLS), None)
WIND_COL  = next((f"{b}_avg" for b in feature_description["sensor_name"]
                  if str(b).startswith("wind_speed_") and f"{b}_avg" in SENSOR_COLS), None)
print(f"Power-curve filter uses: power={POWER_COL}  wind={WIND_COL}")

In [ ]:
# =============================================================================
# PARSED-DATA CACHE
# =============================================================================
# Each event CSV is ~55k rows x ~957 columns and takes 25-45s to parse. Arm A
# reads all 58 once; Arm B reads them twice. That is ~30-60 min of GPU-quota
# time during which the GPU is idle. Build the cache once in a CPU session
# (CPU sessions do not consume GPU quota), save /kaggle/working as a Kaggle
# Dataset, attach it to the GPU notebooks, and every later run skips parsing.

def resolve_cache_dir():
    """Prefer an attached cache dataset; fall back to the working dir."""
    for base in ("/kaggle/input", "/kaggle/working"):
        if not os.path.isdir(base):
            continue
        for dirpath, _, filenames in os.walk(base):
            if "care_cache_manifest.json" in filenames:
                return dirpath
    return os.path.join("/kaggle/working", "care_cache")

CACHE_DIR = resolve_cache_dir()
CACHE_MANIFEST = os.path.join(CACHE_DIR, "care_cache_manifest.json")
CACHE_READY = os.path.exists(CACHE_MANIFEST)

if CACHE_READY:
    _man = json.load(open(CACHE_MANIFEST))
    # A cache built for a different column set would silently change the model
    # input. Refuse it rather than train on the wrong features.
    if set(_man.get("sensor_cols", [])) != set(SENSOR_COLS):
        print(f"Cache at {CACHE_DIR} was built for a different feature set "
              f"({len(_man.get('sensor_cols', []))} cols vs {len(SENSOR_COLS)}). Ignoring it.")
        CACHE_READY = False
    else:
        print(f"Using cache: {CACHE_DIR}  ({_man['n_events']} events, "
              f"built {_man['built_at']})")
if not CACHE_READY:
    print(f"No usable cache. Parsing CSVs directly. "
          f"Run the cache-builder notebook to skip this next time.")

def cache_path(eid):
    return os.path.join(CACHE_DIR, f"event_{eid}.parquet")

def build_cache(out_dir=None, verbose=True):
    """Parse every event CSV once and write it as parquet. CPU session only."""
    import datetime
    out_dir = out_dir or os.path.join("/kaggle/working", "care_cache")
    os.makedirs(out_dir, exist_ok=True)
    t0 = time.time()
    for i, fname in enumerate(csv_files, 1):
        eid = extract_dataset_id(fname)
        df = pd.read_csv(os.path.join(DATASETS_DIR, fname), sep=DATA_SEP,
                         usecols=READ_COLS,
                         dtype={c: DTYPE for c in SENSOR_COLS})
        df["time_stamp"] = pd.to_datetime(df["time_stamp"])
        df = df.sort_values("time_stamp").reset_index(drop=True)
        df.to_parquet(os.path.join(out_dir, f"event_{eid}.parquet"),
                      index=False, compression="snappy")
        if verbose:
            print(f"[{i:>2}/{len(csv_files)}] event {eid:<3} {len(df):>6} rows "
                  f"({time.time()-t0:.0f}s elapsed)")
    json.dump({"sensor_cols": SENSOR_COLS, "descriptive_cols": PRESENT_DESCRIPTIVE,
               "n_events": len(csv_files), "farm": FARM, "sep": DATA_SEP,
               "built_at": datetime.datetime.now().isoformat(timespec="seconds")},
              open(os.path.join(out_dir, "care_cache_manifest.json"), "w"), indent=2)
    print(f"\nCache written to {out_dir} in {(time.time()-t0)/60:.1f} min")
    return out_dir

In [ ]:
# =============================================================================
# PREPROCESSING
# =============================================================================
def read_event_csv(fname):
    """Cache first, else parse only the columns actually used (~226 of 957)."""
    eid = extract_dataset_id(fname)
    if CACHE_READY and os.path.exists(cache_path(eid)):
        return pd.read_parquet(cache_path(eid))      # already sorted, typed
    df = pd.read_csv(os.path.join(DATASETS_DIR, fname), sep=DATA_SEP,
                     usecols=READ_COLS,
                     dtype={c: DTYPE for c in SENSOR_COLS})
    df["time_stamp"] = pd.to_datetime(df["time_stamp"])
    return df.sort_values("time_stamp").reset_index(drop=True)

def compute_normal_mask(df):
    """True = usable as normal-behaviour training data.

    FAQ: status_type_id filtering is the intended use; power-curve filtering is
    a reasonable addition. NOTE this is a TRAINING filter and also the mask for
    pointwise evaluation -- the FAQ excludes abnormal-status timestamps from
    pointwise scoring, because flagging a timestamp the operator already knew
    was abnormal earns no credit.
    """
    m = df["status_type_id"].isin(NORMAL_STATUS_IDS).to_numpy()
    if POWER_COL in df.columns and WIND_COL in df.columns:
        implausible = (df[WIND_COL].between(WIND_SPEED_CUTIN, WIND_SPEED_RATED)
                       & (df[POWER_COL].abs() <= POWER_NEAR_ZERO_THRESH)).to_numpy()
        m = m & ~implausible
    return m

def build_features(df):
    """Sensor frame -> model matrix. Angles become (sin, cos); NaNs stay NaN
    and are imputed AFTER scaling, so a missing value lands on the feature mean
    instead of becoming a large synthetic outlier."""
    parts = [df[LINEAR_COLS].to_numpy(dtype=np.float64)]
    if ANGLE_COLS:
        rad = np.deg2rad(df[ANGLE_COLS].to_numpy(dtype=np.float64))
        parts += [np.sin(rad), np.cos(rad)]
    return np.concatenate(parts, axis=1)

class Standardizer:
    """Fit on training rows only. Impute after scaling (0 == the mean)."""
    def __init__(self):
        self.mean_ = None; self.scale_ = None; self.keep_ = None

    def fit(self, X):
        with np.errstate(invalid="ignore"):
            nan_frac = np.isnan(X).mean(axis=0)
            self.keep_ = nan_frac <= MAX_NAN_FRACTION
            self.mean_ = np.nanmean(np.where(self.keep_, X, np.nan), axis=0)
            self.scale_ = np.nanstd(np.where(self.keep_, X, np.nan), axis=0)
        self.mean_ = np.nan_to_num(self.mean_, nan=0.0)
        self.scale_ = np.where(np.isfinite(self.scale_) & (self.scale_ > 1e-8),
                               self.scale_, 1.0)
        return self

    def transform(self, X):
        Z = (X - self.mean_) / self.scale_
        Z = np.nan_to_num(Z, nan=0.0, posinf=0.0, neginf=0.0)
        return Z[:, self.keep_].astype(DTYPE)

    @property
    def n_out(self):
        return int(self.keep_.sum())

def make_val_split(n, mode=None, frac=VAL_FRACTION, rng=None):
    """Return (train_idx, val_idx) over n time-ordered rows."""
    mode = mode or VAL_SPLIT_MODE
    cut = int(n * (1 - frac))
    if mode == "random":
        rng = rng or np.random.default_rng(SEED)
        idx = rng.permutation(n)
        return idx[:cut], idx[cut:]
    if mode == "chronological":
        return np.arange(cut), np.arange(cut, n)
    block = max(1, int(VAL_BLOCK_HOURS * 6))        # 6 rows per hour at 10-min
    stride = max(2, int(round(1.0 / frac)))         # every stride-th block -> val
    is_val = np.zeros(n, dtype=bool)
    for b in range(int(np.ceil(n / block))):
        if b % stride == stride - 1:
            is_val[b * block:(b + 1) * block] = True
    if is_val.sum() == 0 or is_val.sum() == n:      # too few rows for blocking
        return np.arange(cut), np.arange(cut, n)
    return np.flatnonzero(~is_val), np.flatnonzero(is_val)

def pick_threshold(val_scores, q=None, tag=""):
    """Quantile threshold plus a guard: a val block containing a regime the model
    never saw produces a handful of enormous residuals that drag the quantile up
    and silently disable detection for that event."""
    q = SCORE_QUANTILE if q is None else q
    thr = float(np.quantile(val_scores, q))
    med = float(np.median(val_scores))
    ratio = thr / max(med, 1e-9)
    if ratio > 10:
        print(f"    WARNING{' ' + tag if tag else ''}: threshold is {ratio:.0f}x the "
              f"median validation score (med={med:.2f}, thr={thr:.2f}). The "
              f"validation block likely contains an unseen operating regime.")
    return thr, dict(thr=thr, val_med=med, val_p99=float(np.quantile(val_scores, 0.99)),
                     val_max=float(val_scores.max()), thr_over_median=ratio)

def reconstruction_error(model, X, batch_size=8192):
    """L2 norm of the residual per row. Calls the model directly rather than
    .predict() -- in the per-event loop, .predict() retraces its tf.function on
    every new array shape and floods the log."""
    if len(X) == 0:
        return np.zeros(0, dtype=np.float32)
    out = np.empty(len(X), dtype=np.float32)
    for i in range(0, len(X), batch_size):
        chunk = X[i:i + batch_size]
        rec = model(chunk, training=False).numpy()
        out[i:i + len(chunk)] = np.linalg.norm(chunk - rec, axis=1)
    return out

In [ ]:
# =============================================================================
# ROW FINGERPRINTS  --  leakage instrumentation
# =============================================================================
# The published dataset contains byte-identical rows across different event
# files (an acknowledged side effect of anonymization: "Overlaps across
# different event CSV files are an expected side effect"). A row that is
# `prediction` in one file can be `train` in another. Any notebook that pools
# training data across events therefore trains on its own evaluation rows.
# These fingerprints let each arm MEASURE that rather than assume it away.

def row_fingerprints(df, cols=None, decimals=6):
    cols = cols or SENSOR_COLS
    sub = df[cols].round(decimals)
    return pd.util.hash_pandas_object(sub, index=False).to_numpy()

def leakage_report(train_fp_set, pred_fp_by_event):
    rows = []
    for eid, fps in pred_fp_by_event.items():
        if len(fps) == 0:
            rows.append(dict(event_id=eid, pred_rows=0, leaked_rows=0, leaked_pct=0.0))
            continue
        hit = np.fromiter((f in train_fp_set for f in fps), dtype=bool, count=len(fps))
        rows.append(dict(event_id=eid, pred_rows=int(len(fps)),
                         leaked_rows=int(hit.sum()),
                         leaked_pct=round(100.0 * hit.mean(), 2)))
    rep = pd.DataFrame(rows).sort_values("leaked_pct", ascending=False)
    tot_p, tot_l = rep.pred_rows.sum(), rep.leaked_rows.sum()
    print(f"LEAKAGE: {tot_l:,} / {tot_p:,} prediction rows ({100*tot_l/max(tot_p,1):.2f}%) "
          f"were present in the training corpus")
    print(f"         {(rep.leaked_pct > 0).sum()} / {len(rep)} events affected; "
          f"{(rep.leaked_pct >= 99).sum()} events ~fully leaked")
    return rep

In [ ]:
# =============================================================================
# MODEL  --  paper Table 3 architecture, Wind Farm C
# =============================================================================
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
tf.get_logger().setLevel("ERROR")
tf.random.set_seed(SEED)

def build_autoencoder(n_features, hidden=HIDDEN_LAYERS, noise=NOISE_STD, lr=LEARNING_RATE):
    inp = layers.Input(shape=(n_features,))
    x = layers.GaussianNoise(noise)(inp) if noise > 0 else inp
    for units in hidden:
        x = layers.Dense(units, activation="relu")(x)
    out = layers.Dense(n_features, activation="linear")(x)
    m = models.Model(inp, out)
    m.compile(optimizer=optimizers.Adam(learning_rate=lr), loss="mse")
    return m

def fit_autoencoder(Xtr, Xva, verbose=0, tag=""):
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    model = build_autoencoder(Xtr.shape[1])
    cbs = [callbacks.EarlyStopping(monitor="val_loss", patience=EARLY_STOP_PATIENCE,
                                   restore_best_weights=True, verbose=0)]
    if USE_LR_PLATEAU:
        cbs.append(callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=LR_PLATEAU_FACTOR,
            patience=LR_PLATEAU_PATIENCE, min_lr=1e-6, verbose=0))
    hist = model.fit(Xtr, Xtr, validation_data=(Xva, Xva),
                     batch_size=BATCH_SIZE, epochs=MAX_EPOCHS,
                     callbacks=cbs, verbose=verbose, shuffle=True)
    tr_hist = hist.history["loss"]
    info = dict(tag=tag, epochs_run=len(tr_hist),
                best_val=float(np.min(hist.history["val_loss"])),
                final_train=float(tr_hist[-1]),
                # >0 means training loss ended HIGHER than it started: divergence
                train_drift=float(tr_hist[-1] - min(tr_hist)))
    return model, info

In [ ]:
# =============================================================================
# CARE SCORE  --  per the FAQ
# =============================================================================
# Coverage  : pointwise F-beta on anomalous events. Ground truth = timestamps
#             between event_start and event_end. Abnormal-status timestamps are
#             EXCLUDED from pointwise evaluation (Farm C rule).
# Accuracy  : true-negative rate on normal events, same normal-status mask.
# Earliness : weighted coverage inside the event window, early detections
#             weighted higher. Also restricted to normal-status timestamps so
#             it is measured on the same rows as Coverage.
# Reliability: ONE event-wise F-beta across all events (the FAQ notes it is not
#             averaged, because it is already computed across events).
#             Event decision = criticality counter reaching the threshold.

def fbeta(tp, fp, fn, beta=BETA):
    num = (1 + beta**2) * tp
    den = (1 + beta**2) * tp + (beta**2) * fn + fp
    return float(num / den) if den > 0 else 0.0

def event_ground_truth(times, start, end):
    if start is None or end is None or pd.isna(start) or pd.isna(end):
        return np.zeros(len(times), dtype=int)
    t = pd.to_datetime(times)
    return np.asarray((t >= start) & (t <= end)).astype(int)

def coverage_fbeta(rec, gt):
    m = rec["normal_mask"]; g = gt[m]; p = rec["prediction"][m]
    return fbeta(np.sum((p == 1) & (g == 1)),
                 np.sum((p == 1) & (g == 0)),
                 np.sum((p == 0) & (g == 1)))

def accuracy_care(rec):
    p = rec["prediction"][rec["normal_mask"]]
    tn, fp = np.sum(p == 0), np.sum(p == 1)
    return float(tn / (tn + fp)) if (tn + fp) > 0 else 1.0

def earliness_ws(rec, start, end):
    if start is None or end is None or pd.isna(start) or pd.isna(end):
        return None
    t = pd.to_datetime(rec["time_stamp"])
    inside = np.asarray((t >= start) & (t <= end)) & rec["normal_mask"]
    if inside.sum() == 0:
        return None
    span = (end - start).total_seconds()
    rel = (np.asarray(pd.to_datetime(rec["time_stamp"])[inside]) - np.datetime64(start))
    rel = rel.astype("timedelta64[s]").astype(float) / span if span > 0 else np.zeros(inside.sum())
    w = np.where(rel <= 0.5, 1.0, np.clip(1.0 - (rel - 0.5) / 0.5, 0.0, 1.0))
    return float((w * rec["prediction"][inside]).sum() / w.sum()) if w.sum() > 0 else 0.0

def criticality(status_normal, predictions):
    """FAQ: +1 when an anomaly is flagged on a normal-status timestamp, -1 when
    it is not, unchanged on abnormal-status timestamps. Floored at 0."""
    c = np.zeros(len(predictions) + 1, dtype=np.int32)
    for i in range(1, len(predictions) + 1):
        if not status_normal[i - 1]:
            c[i] = c[i - 1]
        elif predictions[i - 1] == 1:
            c[i] = c[i - 1] + 1
        else:
            c[i] = max(c[i - 1] - 1, 0)
    return c[1:]

def event_detected(rec, thr=CRITICALITY_THRESHOLD):
    c = criticality(rec["normal_mask"], rec["prediction"])
    return (int(c.max() >= thr), int(c.max()) if len(c) else 0) if len(c) else (0, 0)

def evaluate_care(meta, records, verbose=True):
    cov, ws, acc, per_event = [], [], [], []
    y_true, y_pred = [], []
    for eid, m in meta.items():
        rec = records[eid]
        gt = event_ground_truth(rec["time_stamp"], m["event_start"], m["event_end"])
        det, peak = event_detected(rec)
        row = dict(event_id=eid, asset_id=m.get("asset_id"),
                   is_anomaly=int(m["is_anomaly"]), detected=det, peak_criticality=peak,
                   n_pred_rows=int(len(rec["prediction"])),
                   flagged_pct=round(100 * float(rec["prediction"][rec["normal_mask"]].mean())
                                     if rec["normal_mask"].sum() else 0.0, 2))
        if m["is_anomaly"]:
            c = coverage_fbeta(rec, gt); cov.append(c); row["coverage"] = round(c, 4)
            w = earliness_ws(rec, m["event_start"], m["event_end"])
            if w is not None:
                ws.append(w); row["earliness"] = round(w, 4)
        else:
            a = accuracy_care(rec); acc.append(a); row["accuracy"] = round(a, 4)
        y_true.append(int(m["is_anomaly"])); y_pred.append(det)
        per_event.append(row)

    # A component with no supporting events is UNDEFINED, not zero. Treating it
    # as zero silently corrupts any subgroup analysis (e.g. a leaked-only slice
    # with no normal events would report Accuracy 0.0 and trip the Acc<0.5 rule).
    support = dict(coverage=len(cov), earliness=len(ws), accuracy=len(acc))
    C = float(np.mean(cov)) if cov else float("nan")
    E = float(np.mean(ws))  if ws  else float("nan")
    A = float(np.mean(acc)) if acc else float("nan")
    yt, yp = np.array(y_true), np.array(y_pred)
    tp = int(np.sum((yp == 1) & (yt == 1)))
    fp = int(np.sum((yp == 1) & (yt == 0)))
    fn = int(np.sum((yp == 0) & (yt == 1)))
    tn = int(np.sum((yp == 0) & (yt == 0)))
    R = fbeta(tp, fp, fn)

    complete = not (math.isnan(C) or math.isnan(E) or math.isnan(A))
    if not complete:
        care = float("nan")          # undefined on this subset
    elif yp.sum() == 0:
        care = 0.0                   # nothing flagged anywhere
    elif A < 0.5:
        care = A
    else:
        w1, w2, w3, w4 = CARE_WEIGHTS
        care = (w1*C + w2*E + w3*R + w4*A) / (w1 + w2 + w3 + w4)

    out = {"Coverage": C, "Earliness": E, "Reliability": R, "Accuracy": A,
           "CARE": float(care), "support": support,
           "event_confusion": {"tp": tp, "fp": fp, "fn": fn, "tn": tn},
           "n_anomaly_events": int(yt.sum()), "n_normal_events": int((yt == 0).sum())}
    if verbose:
        for k in ["Coverage", "Earliness", "Reliability", "Accuracy", "CARE"]:
            v = out[k]
            print(f"{k:<12} {'n/a  (no supporting events)' if math.isnan(v) else f'{v:.3f}'}")
        print(f"\nEvent-level confusion: tp={tp} fp={fp} fn={fn} tn={tn}")
        print(f"Support: {support['coverage']} anomaly events, "
              f"{support['accuracy']} normal events")
        if complete and care > 0:
            print(f"Accuracy term is {CARE_WEIGHTS[3]*A/(care*sum(CARE_WEIGHTS))*100:.0f}% "
                  f"of the CARE numerator")
        elif complete:
            print("CARE is 0 (no event flagged anywhere in this subset)")
    return out, pd.DataFrame(per_event)

In [ ]:
# =============================================================================
# EFFECTIVE SAMPLE SIZE
# =============================================================================
# CARE averages over events as if they were independent. They are not: several
# events come from the same turbine and share rows outright. Report this next to
# the score so the number is read with the right error bars.
def effective_n_report(meta):
    assets = [m.get("asset_id") for m in meta.values() if m.get("asset_id") is not None]
    if not assets:
        print("No asset_id available; cannot report effective N.")
        return {}
    s = pd.Series(assets)
    counts = s.value_counts()
    print(f"{len(meta)} events span {counts.size} distinct turbines")
    print(f"  turbines contributing >1 event: {(counts > 1).sum()}")
    print(f"  largest single turbine: {counts.max()} events "
          f"({100*counts.max()/len(meta):.1f}% of the 'independent' evaluations)")
    return {"n_events": len(meta), "n_assets": int(counts.size),
            "max_events_per_asset": int(counts.max())}

## Arm A

In [ ]:
# =============================================================================
# ARM A -- EVENT-ISOLATED
# =============================================================================
# One autoencoder per event dataset. Each model sees ONLY its own event's
# training partition; its scaler, its early-stopping split and its threshold are
# all derived from that one file. No row from any other event can reach it.
#
# This follows the paper's structure ("individual autoencoder models per
# turbine") at the strictest granularity, and it is the only configuration in
# which the published duplication cannot contaminate the evaluation.
ARM = "isolated"
print(ARM)

### Train one model per event
~45–60 min on a T4 for 58 models.

In [ ]:
# =============================================================================
# TRAIN + SCORE, ONE EVENT AT A TIME
# =============================================================================
import time
rng = np.random.default_rng(SEED)

dataset_meta, prediction_records, fit_log = {}, {}, []
train_fp_sets, pred_fp_by_event = {}, {}
t0 = time.time()

for k, fname in enumerate(csv_files, 1):
    eid = extract_dataset_id(fname)
    df  = read_event_csv(fname)

    row = event_info[event_info["event_id"].astype(str) == str(eid)].iloc[0]
    is_anom = row["event_label"] == "anomaly"
    dataset_meta[eid] = dict(
        is_anomaly=bool(is_anom),
        event_start=row["event_start"] if is_anom else None,
        event_end=row["event_end"] if is_anom else None,
        asset_id=EVENT_TO_ASSET.get(eid))

    tr_mask   = (df["train_test"] == "train").to_numpy()
    pr_mask   = (df["train_test"] == "prediction").to_numpy()
    norm_mask = compute_normal_mask(df)

    # ---- fingerprints (diagnostic only; never used by the model) ------------
    fp_all = row_fingerprints(df)
    train_fp_sets[eid]    = set(fp_all[tr_mask & norm_mask].tolist())
    pred_fp_by_event[eid] = fp_all[pr_mask]

    # ---- this event's own normal training rows -----------------------------
    X_all = build_features(df)
    X_fit = X_all[tr_mask & norm_mask]
    if len(X_fit) < 2000:
        print(f"[{eid}] only {len(X_fit)} normal training rows -- skipping")
        continue

    # X_fit rows are already in time order (read_event_csv sorts by time_stamp)
    tr_idx, va_idx = make_val_split(len(X_fit), rng=rng)

    scaler = Standardizer().fit(X_fit[tr_idx])          # fit on train slice ONLY
    Xtr, Xva = scaler.transform(X_fit[tr_idx]), scaler.transform(X_fit[va_idx])

    model, info = fit_autoencoder(Xtr, Xva, tag=str(eid))

    # ---- threshold from held-out normal rows of THIS event ------------------
    va_scores = reconstruction_error(model, Xva)
    thr, thr_diag = pick_threshold(va_scores, tag=f"[event {eid}]")
    info.update(event_id=eid, n_train=len(tr_idx), n_val=len(va_idx), **thr_diag)
    fit_log.append(info)

    # ---- score this event's own prediction window --------------------------
    Xp = scaler.transform(X_all[pr_mask])
    sc = reconstruction_error(model, Xp)
    prediction_records[eid] = dict(
        time_stamp=df.loc[pr_mask, "time_stamp"].to_numpy(),
        normal_mask=norm_mask[pr_mask],
        scores=sc,
        prediction=(sc > thr).astype(int))

    print(f"[{k:>2}/{len(csv_files)}] event {eid:<3} "
          f"train={len(tr_idx):>6} epochs={info['epochs_run']:>3} "
          f"val_mse={info['best_val']:.4f} thr={thr:.2f} "
          f"flagged={100*prediction_records[eid]['prediction'].mean():.1f}%")

    del df, X_all, X_fit, Xtr, Xva, Xp, model
    tf.keras.backend.clear_session(); gc.collect()

print(f"\nTrained {len(fit_log)} isolated models in {(time.time()-t0)/60:.1f} min")
fit_log_df = pd.DataFrame(fit_log)

### Verify isolation

In [ ]:
# =============================================================================
# LEAKAGE CHECK -- must be exactly zero for this arm
# =============================================================================
# Each model saw only its own event's training rows, so for event E we check
# E's prediction rows against E's OWN training corpus. Partitions are disjoint
# within a file, so this must come back at 0.00%.
own_leak = []
for eid, fps in pred_fp_by_event.items():
    if eid not in train_fp_sets or len(fps) == 0:
        continue
    s = train_fp_sets[eid]
    hit = np.fromiter((f in s for f in fps), dtype=bool, count=len(fps))
    own_leak.append(dict(event_id=eid, pred_rows=len(fps), leaked=int(hit.sum())))
own_leak = pd.DataFrame(own_leak)
n_hit, n_tot = int(own_leak.leaked.sum()), int(own_leak.pred_rows.sum())
COLLISION_FLOOR = 100.0 * n_hit / max(n_tot, 1)
print(f"Within-file fingerprint collisions: {n_hit} / {n_tot:,} "
      f"({COLLISION_FLOOR:.4f}%)")

# Isolation itself is guaranteed by CONSTRUCTION: training used only
# `tr_mask & norm_mask`, and train_test partitions are disjoint. So any hit here
# is NOT a leak -- it is two different timestamps whose 238 sensor readings are
# identical to 6dp. That happens on flatlined or all-NaN rows during shutdowns.
#
# This number is valuable: it is the empirical FALSE-POSITIVE FLOOR of the
# fingerprint method on this data. Cross-file leakage in the pooled arm is only
# meaningful to the extent it exceeds this floor.
if COLLISION_FLOOR > 1.0:
    print("WARNING: collision floor above 1% -- fingerprints are not discriminative "
          "enough here; inspect degenerate rows before trusting leakage numbers.")
else:
    print("Isolation holds by construction; collision floor is negligible.\n")

# Characterise the colliding rows so the floor is explained, not just reported.
_ex = own_leak[own_leak.leaked > 0].head(3)
if len(_ex):
    print("Events with collisions:", _ex.event_id.tolist())

# For reference: how much leakage a POOLED arm would have incurred on this data.
global_train_fp = set().union(*train_fp_sets.values())
print(f"Counterfactual -- if these {len(train_fp_sets)} training sets had been pooled:")
leak_ref = leakage_report(global_train_fp, pred_fp_by_event)
leak_ref.to_csv("/kaggle/working/leakage_counterfactual.csv", index=False)

### CARE score

In [ ]:
# =============================================================================
# RESULTS
# =============================================================================
results, per_event = evaluate_care(dataset_meta, prediction_records)
print()
results["effective_n"] = effective_n_report(dataset_meta)
results["arm"] = ARM
if "COLLISION_FLOOR" in dir():
    results["fingerprint_collision_floor_pct"] = float(COLLISION_FLOOR)
results["config"] = dict(feature_set="avg_only" if USE_AVG_ONLY else "all",
                         n_features_in=N_FEATURES,
                         hidden=HIDDEN_LAYERS, lr=LEARNING_RATE,
                         batch=BATCH_SIZE, score_quantile=SCORE_QUANTILE,
                         criticality_threshold=CRITICALITY_THRESHOLD)

out_json = f"/kaggle/working/results_{ARM}.json"
with open(out_json, "w") as f:
    json.dump(results, f, indent=2, default=str)
per_event.to_csv(f"/kaggle/working/per_event_{ARM}.csv", index=False)
print(f"\nSaved {out_json} and per_event_{ARM}.csv")
display(per_event)

### Fit diagnostics
Check `epochs_run` — if many models stop at the patience floor, the learning rate is too high for the per-event data volume.

In [ ]:
display(fit_log_df.describe())
display(fit_log_df.sort_values('best_val', ascending=False).head(10))